# Near-Half Court Detector — Training Sweep (Kaggle)

**Setup:**
1. Upload `tennis_vision_training.zip` as a Kaggle Dataset (name it `tennis-vision-training`)
2. Add the dataset to this notebook (+ Add Data → Your Datasets)
3. Enable GPU: Settings → Accelerator → GPU T4 x2 (or GPU P100)
4. Enable Internet: Settings → Internet → On
5. Run All

This notebook runs diagnostic calibration for `lambda_vis`, then an LR sweep.

In [ ]:
import os, zipfile, shutil, glob

# Auto-discover dataset path (handles varying Kaggle mount structures)
matches = glob.glob('/kaggle/input/**/train_near_half_court.py', recursive=True)
if not matches:
    matches = glob.glob('/kaggle/input/**/tennis_vision_training.zip', recursive=True)
    if matches:
        zip_path = matches[0]
        print(f'Found zip at {zip_path}, extracting...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('/kaggle/working')
    else:
        print('Contents of /kaggle/input:')
        for r, d, f in os.walk('/kaggle/input'):
            for fn in f[:3]: print(f'  {os.path.join(r, fn)}')
        raise FileNotFoundError('Cannot find dataset! Click +Add Input in sidebar.')
else:
    dataset_dir = os.path.dirname(matches[0])
    print(f'Dataset found at {dataset_dir}, copying to working dir...')
    for item in os.listdir(dataset_dir):
        s = os.path.join(dataset_dir, item)
        d = f'/kaggle/working/{item}'
        if os.path.isdir(s):
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)

os.chdir('/kaggle/working')
print(f'Working directory: {os.getcwd()}')

# Verify critical files
expected = ['src', 'near_half_train/labels.json', 'near_half_train/crops', 'court_detector.pt', 'train_near_half_court.py']
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ MISSING'
    print(f'  {status}  {p}')

!ls /kaggle/working


In [ ]:
# Install dependencies (torch, torchvision, opencv, numpy are pre-installed on Kaggle)
!pip install -q albumentations pyyaml

In [ ]:
# Verify GPU, imports, and data
import torch

assert torch.cuda.is_available(), 'No GPU detected — enable GPU in Settings → Accelerator!'
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f'GPU: {props.name}  |  VRAM: {vram_gb:.1f} GB')

# Verify dataset imports work
from src.features.court_detect.dataset import create_train_val_datasets, TEST_VIDEO_STEMS, VAL_VIDEO_STEMS
print(f'Val stems: {VAL_VIDEO_STEMS}')
print(f'Test stems: {TEST_VIDEO_STEMS}')

# Quick data sanity check
import json
with open('near_half_train/labels.json') as f:
    data = json.load(f)
labels = data.get('frames', data) if isinstance(data, dict) else data
print(f'Total labelled frames: {len(labels)}')

# Verify a sample image path resolves correctly
sample = labels[0]
img_rel = sample['image_path']  # e.g. 'crops/xxx.jpg'
img_full = os.path.join('near_half_train', img_rel)
assert os.path.isfile(img_full), f'Sample image not found: {img_full}'
print(f'Sample image OK: {img_full}')

# Create datasets to confirm everything wires up
train_ds, val_ds = create_train_val_datasets(
    labels_json='near_half_train/labels.json',
    images_dir='near_half_train',
)
print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}')

In [ ]:
# Create output directories
!mkdir -p logs weights/diagnostic weights/sweep_lr1e5 weights/sweep_lr3e5 weights/sweep_lr1e4

In [ ]:
%%time
# Guard: ensure dataset is available (kernel may have recycled)
import os
if not os.path.exists("train_near_half_court.py"):
    print("⚠️ Kernel recycled! Please re-run the setup cell (Cell 2) first.")
    raise RuntimeError("Kernel recycled — re-run setup cell")

# Diagnostic: calibrate lambda_vis (runs 3 epochs, ~27 min on T4)
import subprocess, re

result = subprocess.run(
    ["python", "train_near_half_court.py",
     "--data-dir", "near_half_train",
     "--labels-json", "near_half_train/labels.json",
     "--pretrained-weights", "court_detector.pt",
     "--diagnostic",
     "--num-workers", "2",
     "--output-dir", "weights/diagnostic"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Parse recommended lambda_vis from output
match = re.search(r"Recommended lambda_vis .* = ([\d.]+)", result.stdout)
if match:
    LAMBDA_VIS = float(match.group(1))
    print(f"\n>>> Auto-captured: LAMBDA_VIS = {LAMBDA_VIS}")
else:
    raise ValueError("Could not parse lambda_vis from diagnostic output!")


In [ ]:
import os, zipfile

# Create incremental zip of all results so far
results_dir = '/kaggle/working/weights'
if os.path.isdir(results_dir):
    zip_path = '/kaggle/working/training_results_after_diagnostic.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(results_dir):
            for fn in files:
                fpath = os.path.join(root, fn)
                arcname = os.path.relpath(fpath, '/kaggle/working')
                zf.write(fpath, arcname)
    sz = os.path.getsize(zip_path) / 1e6
    print(f'✅ Incremental zip: {zip_path} ({sz:.1f} MB)')
else:
    print('⚠️ No weights directory found')


In [ ]:
%%time
# Guard: ensure dataset is available (kernel may have recycled)
import os
if not os.path.exists("train_near_half_court.py"):
    print("⚠️ Kernel recycled! Please re-run the setup cell (Cell 2) first.")
    raise RuntimeError("Kernel recycled — re-run setup cell")

# Sweep 1/3 — backbone-lr = 1e-5
import subprocess
cmd = [
    "python", "train_near_half_court.py",
    "--data-dir", "near_half_train",
    "--labels-json", "near_half_train/labels.json",
    "--pretrained-weights", "court_detector.pt",
    "--backbone-lr", "1e-5",
    "--lambda-vis", str(LAMBDA_VIS),
    "--num-workers", "2",
    "--output-dir", "weights/sweep_lr1e5",
]
print(f"Running with lambda_vis={LAMBDA_VIS}")
proc = subprocess.run(cmd)
assert proc.returncode == 0, f'Training failed with exit code {proc.returncode}'


In [ ]:
import os, zipfile

# Create incremental zip of all results so far
results_dir = '/kaggle/working/weights'
if os.path.isdir(results_dir):
    zip_path = '/kaggle/working/training_results_after_sweep_lr1e5.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(results_dir):
            for fn in files:
                fpath = os.path.join(root, fn)
                arcname = os.path.relpath(fpath, '/kaggle/working')
                zf.write(fpath, arcname)
    sz = os.path.getsize(zip_path) / 1e6
    print(f'✅ Incremental zip: {zip_path} ({sz:.1f} MB)')
else:
    print('⚠️ No weights directory found')


In [ ]:
%%time
# Guard: ensure dataset is available (kernel may have recycled)
import os
if not os.path.exists("train_near_half_court.py"):
    print("⚠️ Kernel recycled! Please re-run the setup cell (Cell 2) first.")
    raise RuntimeError("Kernel recycled — re-run setup cell")

# Sweep 2/3 — backbone-lr = 3e-5
import subprocess
cmd = [
    "python", "train_near_half_court.py",
    "--data-dir", "near_half_train",
    "--labels-json", "near_half_train/labels.json",
    "--pretrained-weights", "court_detector.pt",
    "--backbone-lr", "3e-5",
    "--lambda-vis", str(LAMBDA_VIS),
    "--num-workers", "2",
    "--output-dir", "weights/sweep_lr3e5",
]
print(f"Running with lambda_vis={LAMBDA_VIS}")
proc = subprocess.run(cmd)
assert proc.returncode == 0, f'Training failed with exit code {proc.returncode}'


In [ ]:
import os, zipfile

# Create incremental zip of all results so far
results_dir = '/kaggle/working/weights'
if os.path.isdir(results_dir):
    zip_path = '/kaggle/working/training_results_after_sweep_lr3e5.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(results_dir):
            for fn in files:
                fpath = os.path.join(root, fn)
                arcname = os.path.relpath(fpath, '/kaggle/working')
                zf.write(fpath, arcname)
    sz = os.path.getsize(zip_path) / 1e6
    print(f'✅ Incremental zip: {zip_path} ({sz:.1f} MB)')
else:
    print('⚠️ No weights directory found')


In [ ]:
%%time
# Guard: ensure dataset is available (kernel may have recycled)
import os
if not os.path.exists("train_near_half_court.py"):
    print("⚠️ Kernel recycled! Please re-run the setup cell (Cell 2) first.")
    raise RuntimeError("Kernel recycled — re-run setup cell")

# Sweep 3/3 — backbone-lr = 1e-4
import subprocess
cmd = [
    "python", "train_near_half_court.py",
    "--data-dir", "near_half_train",
    "--labels-json", "near_half_train/labels.json",
    "--pretrained-weights", "court_detector.pt",
    "--backbone-lr", "1e-4",
    "--lambda-vis", str(LAMBDA_VIS),
    "--num-workers", "2",
    "--output-dir", "weights/sweep_lr1e4",
]
print(f"Running with lambda_vis={LAMBDA_VIS}")
proc = subprocess.run(cmd)
assert proc.returncode == 0, f'Training failed with exit code {proc.returncode}'


In [ ]:
import os, zipfile

# Create incremental zip of all results so far
results_dir = '/kaggle/working/weights'
if os.path.isdir(results_dir):
    zip_path = '/kaggle/working/training_results_after_sweep_lr1e4.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(results_dir):
            for fn in files:
                fpath = os.path.join(root, fn)
                arcname = os.path.relpath(fpath, '/kaggle/working')
                zf.write(fpath, arcname)
    sz = os.path.getsize(zip_path) / 1e6
    print(f'✅ Incremental zip: {zip_path} ({sz:.1f} MB)')
else:
    print('⚠️ No weights directory found')


In [ ]:
# Compare results across sweep runs
import torch, glob, os

sweep_configs = [
    ('1e-5', 'weights/sweep_lr1e5'),
    ('3e-5', 'weights/sweep_lr3e5'),
    ('1e-4', 'weights/sweep_lr1e4'),
]

rows = []
for lr_label, out_dir in sweep_configs:
    # Find the best checkpoint file
    candidates = sorted(glob.glob(os.path.join(out_dir, '*best*')))
    if not candidates:
        candidates = sorted(glob.glob(os.path.join(out_dir, '*.pt')))
    if not candidates:
        rows.append({'lr': lr_label, 'file': 'NO CHECKPOINT FOUND'})
        continue

    ckpt_path = candidates[0]
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    row = {'lr': lr_label, 'file': os.path.basename(ckpt_path)}
    # Extract metrics if stored in checkpoint
    for key in ('val_loss', 'best_val_loss', 'epoch', 'val_kp_error', 'val_vis_acc'):
        if isinstance(ckpt, dict) and key in ckpt:
            val = ckpt[key]
            row[key] = f'{val:.4f}' if isinstance(val, float) else val
    rows.append(row)

# Print comparison table
if rows:
    all_keys = list(dict.fromkeys(k for r in rows for k in r.keys()))
    header = ' | '.join(f'{k:>14s}' for k in all_keys)
    print(header)
    print('-' * len(header))
    for r in rows:
        vals = [str(r.get(k, '—')) for k in all_keys]
        print(' | '.join(f'{v:>14s}' for v in vals))

In [ ]:
# Package results for download from Kaggle Output tab
import shutil, os

results_dir = '/kaggle/working/results'
os.makedirs(results_dir, exist_ok=True)

sweep_dirs = [
    ('diagnostic', 'weights/diagnostic'),
    ('sweep_lr1e5', 'weights/sweep_lr1e5'),
    ('sweep_lr3e5', 'weights/sweep_lr3e5'),
    ('sweep_lr1e4', 'weights/sweep_lr1e4'),
]

for name, src_dir in sweep_dirs:
    if os.path.isdir(src_dir):
        dest = os.path.join(results_dir, name)
        shutil.copytree(src_dir, dest, dirs_exist_ok=True)
        print(f"  ✅ {name}")
    else:
        print(f"  ❌ {name} not found")

if os.path.isdir('logs'):
    shutil.copytree('logs', os.path.join(results_dir, 'logs'), dirs_exist_ok=True)

shutil.make_archive('/kaggle/working/training_results', 'zip', results_dir)
print('✅ Results packaged at /kaggle/working/training_results.zip')
print('Download from the Output tab on the right →')
